# 📡 12 — Online Evaluation, Alerts and Closing the Loop

Offline evaluation catches what your dataset represents. Production is where you find out
what it didn't. This notebook simulates a day of traffic, works out which metrics survive
without reference answers, and closes the loop back to notebook 03.

## Learning Objectives
In this notebook, you will learn:
1. **Which metrics survive online** - faithfulness, goal accuracy without reference, topic adherence; and which are structurally offline-only
2. **Sampling strategy** - safety assertions on 100% of traffic, judge sampling on 1-2%
3. **Five alerts** - cost at 3x trailing median, citation-free answer rate, tool-status mix shifting toward `empty`, approval decision time, loop rate per agent
4. **Incident drills** - `empty_index` and `retry_loop`: which alert fires first, which stays silent
5. **Canary and rollback** - new versus old on the same traffic, paired win rate
6. **The loop** - sampled traces -> notebook 09 for labels -> notebook 03 for new goldens -> drift check

## Prerequisites
- Notebooks 09, 10 and 11

---
## 🗺️ The concept

```
  OFFLINE                          ONLINE
  ------------------------------   ------------------------------
  has reference answers            has none, ever
  runs on 40 curated cases         runs on everything
  catches large, known failures    catches drift and the unrepresented
  blocks a release                 pages a human

  WHAT STILL WORKS WITHOUT A REFERENCE
    faithfulness            answer vs. its OWN logged context
    goal accuracy           (without reference)
    topic adherence         did it stay in scope
    trajectory rules        deterministic, free, 100% of traffic
    paired canary           new vs. old on the same inputs
    delayed human labels    calibration, days later

  THE LOOP
    production trace -> sampled -> nb09 human label -> nb03 new golden
                                        |
                                        +-> nb11 gate gets harder
```

### Key Concepts:
- **Stop scoring correctness; start scoring consistency and shape.** Without a reference you
  cannot ask "is this right?", but you can ask "is this grounded in what it retrieved?" and
  "is this better than the build we already shipped?"
- **Cheap checks run on everything; expensive ones are sampled.** Deterministic assertions
  cost nothing, so they cover 100% of traffic. Judges cost money, so they see 1-2%.
- **An alert that never fires is not reassurance.** Two of the five below stay silent during a
  real incident, and knowing *which* is the point of the drills.

---
## 🔨 Build it

### 0.1 ⚙️ Setup

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: the shipped build, and the gate that let it through
# ============================================================================
import os
import json
import math
import random
import warnings
import statistics
from collections import defaultdict

warnings.filterwarnings("ignore")

DATA_DIR = os.path.abspath("data")


def load(name, required=True):
    path = os.path.join(DATA_DIR, name)
    if not os.path.exists(path):
        if required:
            raise FileNotFoundError(name + " is missing — run its notebook first.")
        return None
    with open(path, encoding="utf-8") as f:
        return json.load(f)


BASELINE = load("baseline_report.json")
GATE_RESULT = load("gate_result.json")
TRAJECTORIES = load("trajectories.json")

with open(os.path.join(DATA_DIR, "golden_set.json"), encoding="utf-8") as f:
    GOLDEN = json.load(f)

print("✅ Setup complete")
print("🚦 shipped build : %s (%s)"
      % (GATE_RESULT["candidate"],
         "allowed" if GATE_RESULT["decision"]["allowed"] else "blocked"))
print("📊 offline baseline resolution: %.3f"
      % BASELINE["scores"]["resolution_rate"])

---
## 🌐 Part 1: A day of production traffic

Production differs from the golden set in three ways that matter, and the simulator below
reproduces all three deliberately:

1. **The mix is not your suite's mix.** Real traffic is mostly easy, repetitive billing
   questions; your golden set is deliberately hard.
2. **There are no reference answers.** Ever.
3. **There is a long tail your dataset never represented** — the questions that make online
   evaluation worth doing at all.

In [ ]:
# ============================================================================
# TRAFFIC SIMULATOR: a day of conversations, with production's real mix
# ============================================================================
# Seeded so every number in this notebook reproduces. The `unrepresented` slice
# is the whole reason online evaluation exists: the golden set has nothing
# like it, so offline evaluation is structurally blind to it.

random.seed(12)

TRAFFIC_MIX = {
    "billing":   0.42,
    "account":   0.26,
    "knowledge": 0.22,
    "escalation": 0.06,
    "unrepresented": 0.04,       # nothing in the golden set looks like these
}
LANG_MIX = {"en": 0.70, "es": 0.18, "hi": 0.12}


def weighted_choice(mix, rng):
    roll = rng.random()
    cumulative = 0.0
    for key, weight in mix.items():
        cumulative += weight
        if roll <= cumulative:
            return key
    return list(mix)[-1]


def simulate_day(n=1200, seed=12, faults=None):
    """One dict per conversation. No reference answers — that is the point."""
    rng = random.Random(seed)
    faults = faults or {}
    events = []

    for i in range(n):
        intent = weighted_choice(TRAFFIC_MIX, rng)
        lang = weighted_choice(LANG_MIX, rng)

        # --- retrieval -----------------------------------------------------
        if faults.get("empty_index"):
            n_contexts = 0
        elif intent == "unrepresented":
            n_contexts = rng.choice([0, 1])
        else:
            n_contexts = rng.choice([2, 3, 3])

        cited = n_contexts > 0 and rng.random() < 0.93

        # --- tool calls ----------------------------------------------------
        steps = []
        n_calls = {"billing": 2, "account": 2, "knowledge": 1,
                   "escalation": 1, "unrepresented": 1}[intent]
        for _ in range(n_calls):
            status = "ok"
            if faults.get("tool_empty") and rng.random() < 0.5:
                status = "empty"
            elif rng.random() < 0.04:
                status = rng.choice(["empty", "unavailable"])
            steps.append({"tool": "get_invoice" if intent == "billing"
                          else "get_account", "status": status})

        if faults.get("retry_loop") and intent == "billing":
            steps = steps + [dict(steps[0]) for _ in range(3)]

        tokens = 700 + 90 * len(steps) + rng.randint(-80, 180)
        if faults.get("verbose_prompt"):
            tokens *= 3

        events.append({
            "id": "T%05d" % i,
            "intent": intent,
            "lang": lang,
            "n_contexts": n_contexts,
            "cited": cited,
            "steps": steps,
            "tokens": tokens,
            "cost_usd": round(tokens / 1000.0 * 0.0006, 6),
            "resolved": (n_contexts > 0 or intent in ("billing", "account"))
                        and rng.random() < 0.88,
            "approval_seconds": (rng.uniform(15, 140)
                                 if intent == "billing" and rng.random() < 0.3
                                 else None),
        })
    return events


TRAFFIC = simulate_day()

print("🌐 conversations today: %d" % len(TRAFFIC))
by_intent = defaultdict(int)
for e in TRAFFIC:
    by_intent[e["intent"]] += 1
for key in sorted(by_intent):
    print("   %-16s %5d  (%.0f%%)" % (key, by_intent[key],
                                      100.0 * by_intent[key] / len(TRAFFIC)))
print()
print("⚠️  %d conversations are of a kind your golden set contains ZERO examples"
      % by_intent["unrepresented"])
print("    of. Offline evaluation cannot say anything about them, at any budget.")

---
## 🔎 Part 2: Which metrics survive without a reference?

This is the table worth memorising. The right-hand column is what your production dashboard can
actually contain.

| Metric | Needs a reference? | Online? | Why |
|---|---|---|---|
| Context recall | **Yes** | ❌ | Needs the set of documents that *should* have been retrieved |
| Context precision | **Yes** | ❌ | Same |
| Answer correctness | **Yes** | ❌ | Needs the right answer |
| Tool correctness | **Yes** | ❌ | Needs `expected_tools` |
| **Faithfulness** | No | ✅ | Answer versus its own logged context |
| **Goal accuracy (without reference)** | No | ✅ | Judge infers the goal from the conversation |
| **Topic adherence** | No | ✅ | Scope is defined by policy, not by a golden answer |
| **Trajectory/safety rules** | No | ✅ **100%** | Deterministic and free |
| **Paired canary** | No | ✅ | Compares two builds, not a build to an ideal |

Notice the shape of it: **everything that needs an answer key dies at deployment.** Notebook 08
measured exactly this gap with its with-reference / without-reference rubric pair.

In [ ]:
# ============================================================================
# SAMPLING: cheap checks on everything, judges on 1-2%
# ============================================================================
# The arithmetic is worth doing out loud, because it is usually the argument
# that gets judge sampling approved.

JUDGE_SAMPLE_RATE = 0.02
JUDGE_COST_PER_CALL = 0.0015          # a small judge, one call per conversation

n = len(TRAFFIC)
sampled = [e for e in TRAFFIC if random.Random(hash(e["id"]) & 0xFFFF).random()
           < JUDGE_SAMPLE_RATE]

print("🔎 deterministic checks : %d conversations (100%%, $0.00)" % n)
print("🔎 judged               : %d conversations (%.1f%%, $%.2f/day)"
      % (len(sampled), 100.0 * len(sampled) / n,
         len(sampled) * JUDGE_COST_PER_CALL))
print("   judging everything would cost $%.2f/day, $%.0f/year"
      % (n * JUDGE_COST_PER_CALL, n * JUDGE_COST_PER_CALL * 365))
print()
print("📌 The sample is for TREND, not for catching individual failures. Anything")
print("   that must be caught every single time has to be a deterministic rule —")
print("   which is why notebook 07's policies read state instead of prose.")

---
## 🚨 Part 3: Five alerts

Each alert is a function over a window of traffic. They are deliberately **not** quality
metrics — they are *shape* metrics, because shape is what you can measure without an answer key.

| Alert | Fires when | Catches |
|---|---|---|
| Cost spike | Cost per conversation > 3× trailing median | `verbose_prompt`, `retry_loop` |
| Citation-free rate | Share of answers citing nothing exceeds a floor | `empty_index`, retrieval outages |
| Tool status mix | `empty` share shifts materially | A data source going quietly stale |
| Approval decision time | Median falls below a floor | Rubber-stamping (notebook 09) |
| Loop rate | Repeated identical calls per conversation | `retry_loop` |

In [ ]:
# ============================================================================
# THE FIVE ALERTS: each one a plain function over a window
# ============================================================================
# Thresholds are in one dict, the same discipline as notebook 11's GATE_CONFIG.

ALERT_CONFIG = {
    "cost_spike_multiple": 3.0,
    "citation_free_rate_max": 0.15,
    "tool_empty_share_max": 0.20,
    "approval_median_seconds_min": 5.0,
    "loop_rate_max": 0.05,
}


def cost_per_conversation(window):
    return statistics.mean(e["cost_usd"] for e in window) if window else 0.0


def alert_cost_spike(window, trailing_median):
    now = cost_per_conversation(window)
    limit = trailing_median * ALERT_CONFIG["cost_spike_multiple"]
    return {"name": "cost_spike", "value": now, "limit": limit,
            "fired": trailing_median > 0 and now > limit}


def alert_citation_free(window):
    rate = statistics.mean(0.0 if e["cited"] else 1.0 for e in window)
    limit = ALERT_CONFIG["citation_free_rate_max"]
    return {"name": "citation_free_rate", "value": rate, "limit": limit,
            "fired": rate > limit}


def alert_tool_status_mix(window):
    statuses = [s["status"] for e in window for s in e["steps"]]
    share = (statistics.mean(1.0 if s == "empty" else 0.0 for s in statuses)
             if statuses else 0.0)
    limit = ALERT_CONFIG["tool_empty_share_max"]
    return {"name": "tool_empty_share", "value": share, "limit": limit,
            "fired": share > limit}


def alert_approval_time(window):
    times = [e["approval_seconds"] for e in window if e["approval_seconds"]]
    median = statistics.median(times) if times else None
    limit = ALERT_CONFIG["approval_median_seconds_min"]
    return {"name": "approval_decision_time", "value": median, "limit": limit,
            "fired": median is not None and median < limit}


def alert_loop_rate(window):
    looping = 0
    for e in window:
        signatures = [s["tool"] for s in e["steps"]]
        if any(signatures.count(x) > 2 for x in set(signatures)):
            looping += 1
    rate = looping / len(window) if window else 0.0
    limit = ALERT_CONFIG["loop_rate_max"]
    return {"name": "loop_rate", "value": rate, "limit": limit,
            "fired": rate > limit}


def run_alerts(window, trailing_median):
    return [alert_cost_spike(window, trailing_median),
            alert_citation_free(window),
            alert_tool_status_mix(window),
            alert_approval_time(window),
            alert_loop_rate(window)]


def print_alerts(alerts, title):
    print("── %s ──" % title)
    for a in alerts:
        value = "n/a" if a["value"] is None else "%.4f" % a["value"]
        print("   %-24s %-10s limit %-10.4f %s"
              % (a["name"], value, a["limit"], "🚨 FIRED" if a["fired"] else "ok"))
    print()


TRAILING_MEDIAN = cost_per_conversation(TRAFFIC)
print_alerts(run_alerts(TRAFFIC, TRAILING_MEDIAN), "healthy day")

---
## 🔥 Part 4: Incident drills

Two faults, in production this time. The question is not *"does something break?"* — it is
**which alert fires, how fast, and which one stays quiet while the incident is ongoing.**

An alert that stays silent during a real incident is not a bad alert. It is an alert measuring
something else, and knowing which is what lets you stop staring at the wrong dashboard.

In [ ]:
# ============================================================================
# DRILL 1: empty_index in production — retrieval returns nothing, silently
# ============================================================================
broken = simulate_day(faults={"empty_index": True}, seed=21)

healthy_alerts = run_alerts(TRAFFIC, TRAILING_MEDIAN)
broken_alerts = run_alerts(broken, TRAILING_MEDIAN)

print("%-24s %14s %14s" % ("alert", "healthy", "empty_index"))
for h, b in zip(healthy_alerts, broken_alerts):
    hv = "n/a" if h["value"] is None else "%.3f" % h["value"]
    bv = "n/a" if b["value"] is None else "%.3f" % b["value"]
    print("%-24s %8s %5s %8s %5s" % (
        h["name"], hv, "🚨" if h["fired"] else "ok",
        bv, "🚨" if b["fired"] else "ok"))

print()
resolved_h = statistics.mean(1.0 if e["resolved"] else 0.0 for e in TRAFFIC)
resolved_b = statistics.mean(1.0 if e["resolved"] else 0.0 for e in broken)
print("   resolution rate: %.3f -> %.3f (%+.3f)"
      % (resolved_h, resolved_b, resolved_b - resolved_h))
print()
print("⚠️  The citation-free alert fires immediately — it is a counting rule over")
print("    traffic you already log, so it is minutes old, not hours. Resolution")
print("    rate barely moves, because billing and account questions still")
print("    'succeed' without any documentation behind them.")

In [ ]:
# ============================================================================
# DRILL 2: retry_loop in production — expensive, and invisible to quality
# ============================================================================
looped = simulate_day(faults={"retry_loop": True}, seed=22)
looped_alerts = run_alerts(looped, TRAILING_MEDIAN)

print("%-24s %14s %14s" % ("alert", "healthy", "retry_loop"))
for h, l in zip(healthy_alerts, looped_alerts):
    hv = "n/a" if h["value"] is None else "%.3f" % h["value"]
    lv = "n/a" if l["value"] is None else "%.3f" % l["value"]
    print("%-24s %8s %5s %8s %5s" % (
        h["name"], hv, "🚨" if h["fired"] else "ok",
        lv, "🚨" if l["fired"] else "ok"))

print()
print("   resolution rate: %.3f -> %.3f"
      % (statistics.mean(1.0 if e["resolved"] else 0.0 for e in TRAFFIC),
         statistics.mean(1.0 if e["resolved"] else 0.0 for e in looped)))
print("   cost per conversation: $%.5f -> $%.5f (%.1fx)"
      % (cost_per_conversation(TRAFFIC), cost_per_conversation(looped),
         cost_per_conversation(looped) / cost_per_conversation(TRAFFIC)))
print()
print("⚠️  Cost and loop-rate fire. Citation-free stays quiet. Resolution rate is")
print("    UNCHANGED — customers are being helped correctly, repeatedly, at")
print("    several times the price. No quality dashboard would show this.")

---
## 🐤 Part 5: The canary — the only online question that is properly answerable

Without a reference you cannot ask *"is this build good?"*. You can ask **"is it better than the
one already running?"** — and that is the actual decision anyway.

Route a slice of traffic to the new build, the same slice's inputs to the old one, and count
paired wins. `b` and `c` below are the same two cells McNemar used in notebook 10, which is not
a coincidence: it is the same question about the same kind of paired data.

In [ ]:
# ============================================================================
# CANARY: paired comparison on the same inputs, then a rollback decision
# ============================================================================

def run_canary(traffic, share=0.10, candidate_better=0.06, seed=31):
    """Return (both_ok, old_only, new_only, both_bad) over the canary slice."""
    rng = random.Random(seed)
    slice_ = [e for e in traffic if rng.random() < share]

    both_ok = old_only = new_only = both_bad = 0
    for e in slice_:
        old_ok = e["resolved"]
        # The candidate is the same conversation with a small shift in outcome.
        if rng.random() < abs(candidate_better):
            new_ok = candidate_better > 0
        else:
            new_ok = old_ok
        if old_ok and new_ok:
            both_ok += 1
        elif old_ok and not new_ok:
            old_only += 1
        elif new_ok and not old_ok:
            new_only += 1
        else:
            both_bad += 1
    return both_ok, old_only, new_only, both_bad


def mcnemar_p(c, d):
    """Identical to notebook 10's. c = regressions, d = fixes."""
    if c + d == 0:
        return 1.0
    statistic = max(0.0, (abs(c - d) - 1) ** 2 / (c + d))
    return math.erfc(math.sqrt(statistic / 2.0))


def canary_verdict(traffic, candidate_better, label, seed=31):
    both_ok, regressions, fixes, both_bad = run_canary(
        traffic, candidate_better=candidate_better, seed=seed)
    n = both_ok + regressions + fixes + both_bad
    p = mcnemar_p(regressions, fixes)
    changed = regressions + fixes
    win_rate = (fixes / changed) if changed else float("nan")

    if p < 0.05 and regressions > fixes:
        verdict = "🔴 ROLL BACK"
    elif p < 0.05 and fixes > regressions:
        verdict = "🟢 PROMOTE"
    else:
        verdict = "🟡 KEEP WATCHING"

    print("%-22s n=%-5d fixes=%-4d regressions=%-4d win=%-6s p=%.4f  %s"
          % (label, n, fixes, regressions,
             "n/a" if changed == 0 else "%.2f" % win_rate, p, verdict))
    return verdict


print("%-22s %s" % ("canary", "paired outcome on the same inputs"))
canary_verdict(TRAFFIC, 0.06, "slightly better")
canary_verdict(TRAFFIC, -0.08, "slightly worse")
canary_verdict(TRAFFIC, 0.005, "indistinguishable")
print()
print("📌 'KEEP WATCHING' is a real answer, not a failure to decide. With a small")
print("   canary slice most differences are not resolvable, and the honest move")
print("   is to keep the canary running rather than to promote on a coin flip.")

---
## 🔁 Part 6: Closing the loop

Everything above produces signal. The loop is what turns signal into a **harder suite**, which
is the only durable outcome:

```
  production trace
        |
        v  sampled: judge disagreement, alert-adjacent, unrepresented intents
  notebook 09   human label
        |
        v
  notebook 03   new golden case (with strata tags)
        |
        v
  notebook 11   the gate now blocks on a failure that once shipped
```

The rule worth keeping: **every incident ends by adding a golden case.** An incident that does
not change the suite will happen again, and the second time nobody will remember the first.

In [ ]:
# ============================================================================
# THE LOOP: choose which production traces become tomorrow's golden cases
# ============================================================================
# Not a random sample. These three buckets are where the suite is weakest, which
# is exactly where new cases are worth the annotation cost.

def select_for_labelling(traffic, per_bucket=8, seed=41):
    rng = random.Random(seed)
    buckets = {
        # 1. Intents the golden set has no examples of at all.
        "unrepresented": [e for e in traffic if e["intent"] == "unrepresented"],
        # 2. Answers with no citation — the shape that precedes a hallucination.
        "citation_free": [e for e in traffic if not e["cited"]],
        # 3. Tool trouble — where result handling (notebook 05) gets decided.
        "tool_not_ok": [e for e in traffic
                        if any(s["status"] != "ok" for s in e["steps"])],
    }
    selected = {}
    for name, items in buckets.items():
        rng.shuffle(items)
        selected[name] = items[:per_bucket]
    return selected


SELECTED = select_for_labelling(TRAFFIC)

print("%-18s %10s %12s" % ("bucket", "available", "queued"))
for name, items in SELECTED.items():
    available = sum(1 for e in TRAFFIC if (
        e["intent"] == "unrepresented" if name == "unrepresented"
        else (not e["cited"]) if name == "citation_free"
        else any(s["status"] != "ok" for s in e["steps"])))
    print("%-18s %10d %12d" % (name, available, len(items)))

queue_path = os.path.join(DATA_DIR, "label_queue.json")
with open(queue_path, "w", encoding="utf-8") as f:
    json.dump({"selected_at": "simulated-day-1",
               "buckets": {k: [e["id"] for e in v] for k, v in SELECTED.items()}},
              f, ensure_ascii=False, indent=2)

print()
print("💾 Wrote", queue_path, "— notebook 09's panel reads a queue in this shape.")
print()
print("📌 Note what is NOT in the queue: a random sample of healthy traffic. It")
print("   would be cheaper to label and would teach you nothing you do not")
print("   already know from the golden set.")

In [ ]:
# ============================================================================
# DRIFT CHECK: is production still shaped like the suite you gate on?
# ============================================================================
# A golden set is a claim about what traffic looks like. Traffic moves. When it
# moves far enough, your per-stratum floors are protecting the wrong segments.

golden_mix = defaultdict(int)
for case in GOLDEN["cases"]:
    golden_mix[case["strata"]["lang"]] += 1

production_mix = defaultdict(int)
for e in TRAFFIC:
    production_mix[e["lang"]] += 1


def normalise(counts):
    total = sum(counts.values())
    return {k: v / total for k, v in counts.items()}


golden_share = normalise(golden_mix)
production_share = normalise(production_mix)

print("%-8s %12s %14s %10s" % ("lang", "golden set", "production", "drift"))
total_variation = 0.0
for key in sorted(set(golden_share) | set(production_share)):
    g = golden_share.get(key, 0.0)
    p = production_share.get(key, 0.0)
    total_variation += abs(g - p)
    print("%-8s %11.1f%% %13.1f%% %+9.1f" % (key, 100 * g, 100 * p,
                                             100 * (p - g)))

print()
print("📏 total variation distance: %.3f" % (total_variation / 2.0))
print()
if total_variation / 2.0 > 0.15:
    print("⚠️  Production no longer looks like the golden set. The per-stratum")
    print("    floors in notebook 11 are weighted for a traffic mix you do not")
    print("    have any more — rebalance the suite, or accept that you are")
    print("    over-protecting a segment that shrank.")
else:
    print("✅ The suite still resembles production closely enough to gate on.")

---
## ▶️ See it work

A day of traffic, five alerts, a canary verdict and a label queue — the whole online loop.

In [ ]:
# ============================================================================
# SEE IT WORK: the production dashboard, such as it can honestly be
# ============================================================================
print("── traffic ──")
print("   conversations      : %d" % len(TRAFFIC))
print("   resolution rate    : %.3f (NO reference answers involved)"
      % statistics.mean(1.0 if e["resolved"] else 0.0 for e in TRAFFIC))
print("   cost/conversation  : $%.5f" % cost_per_conversation(TRAFFIC))
print("   judged sample      : %d (%.1f%%)" % (len(sampled),
                                               100.0 * len(sampled) / len(TRAFFIC)))
print()
print_alerts(healthy_alerts, "alerts")

print("── offline vs online ──")
print("   offline resolution (golden set) : %.3f"
      % BASELINE["scores"]["resolution_rate"])
print("   online  resolution (production) : %.3f"
      % statistics.mean(1.0 if e["resolved"] else 0.0 for e in TRAFFIC))
print()
print("   These two numbers are NOT comparable and should never be plotted on")
print("   the same axis. The golden set is deliberately hard; production is")
print("   mostly easy. A gap is expected — a gap that CHANGES is the signal.")

---
## 💥 Break it

| Fault | What it should catch |
|---|---|
| `empty_index in production` | The citation-free answer rate alert fires within minutes; the quality dashboard stays green for hours |
| `retry_loop in production` | The cost alert fires and the loop-rate alert fires; no quality alert fires at all |

Both drills ran in Part 4. The table below is the summary worth keeping on a wall: **which
alert is load-bearing for which incident.**

In [ ]:
# ============================================================================
# THE ALERT COVERAGE MATRIX — including the gaps
# ============================================================================
INCIDENTS = [
    ("healthy", {}),
    ("empty_index", {"empty_index": True}),
    ("retry_loop", {"retry_loop": True}),
    ("verbose_prompt", {"verbose_prompt": True}),
    ("tool source stale", {"tool_empty": True}),
]

ALERT_NAMES = [a["name"] for a in healthy_alerts]

print("%-20s %s" % ("incident", " ".join("%-22s" % n for n in ALERT_NAMES)))
rows = []
for label, faults in INCIDENTS:
    day = simulate_day(faults=faults, seed=51)
    alerts = run_alerts(day, TRAILING_MEDIAN)
    fired = {a["name"]: a["fired"] for a in alerts}
    rows.append((label, fired))
    print("%-20s %s" % (label, " ".join("%-22s" % ("🚨 FIRED" if fired[n] else "·")
                                        for n in ALERT_NAMES)))

print()
never_fired = [n for n in ALERT_NAMES
               if not any(fired[n] for label, fired in rows if label != "healthy")]
print("📌 Alerts that never fired in any drill: %s" % (never_fired or "none"))
print("   That is not proof they are useless — the approval-time alert catches")
print("   rubber-stamping, which none of these drills simulate. But an alert you")
print("   have never SEEN fire is an alert you do not know is wired up.")
print()
print("⚠️  Run a drill for every alert you own, on a schedule. An alert whose")
print("    last confirmed firing was during its own code review is decoration.")

---
## 🧪 Try it yourself

**Exercise 1 — Add the alert this notebook is missing.**
Nothing here watches **judge drift**, even though notebook 09 showed kappa collapsing while raw
agreement barely moved. Add a sixth alert that recomputes kappa against the most recent human
labels and fires when it falls below a floor. The hard part is the window: human labels arrive
days late, so decide what the alert does on the days when there are none.

**Exercise 2 — Make the canary decision properly.**
`canary_verdict` promotes on `p < 0.05`. That treats a regression and a missed improvement as
equally costly. Add a minimum sample size and a minimum effect size, then answer: how long does
the canary need to run, at this traffic volume, to detect a 3-point regression? Notebook 10's
sample-size arithmetic already gives you the number.

**Exercise 3 — Close the loop for real.**
Take five ids from `label_queue.json`, label them with notebook 09's panel, write them as golden
cases in notebook 03's `case()` format, bump `GOLDEN_SET_VERSION`, and re-run notebook 11. Then
check: does the gate now block a build that previously shipped? If not, your new cases were too
easy, which is itself the finding.

---
## 📝 Recap and interview angle

### Offline and online, side by side

| | Offline (notebooks 04–11) | Online (this notebook) |
|---|---|---|
| Reference answers | Yes | **Never** |
| Coverage | 40 curated cases | All traffic |
| Catches | Large, known, deterministic regressions | Drift, and everything the suite never represented |
| Consequence | Blocks a release | Pages a human |
| Cost | Bounded, paid once per build | Continuous — hence sampling |
| Cheapest signal | Deterministic assertions | The same assertions, on 100% of traffic |

### The ecosystem, briefly

| Tool | What it is genuinely good at here |
|---|---|
| **RAGAS** | RAG-specific metrics; the with/without-reference split is unusually honest |
| **DeepEval** | Component-level metrics, `assert_test` in CI, tracing via `@observe` |
| **Langfuse** | Trace storage, annotation queues, user-feedback scores — the notebook-09 loop |
| **MLflow / LangSmith** | Experiment tracking and trace comparison across builds |

None of them ships the two things this course spent the most time on: the **final-state check**
and the **gate**. Those are yours, because only you know what your database should look like and
what "good enough to ship" means for your product.

### Produces
| Artifact | Consumed by |
|---|---|
| `data/online_report.json` | Your dashboard — and the next iteration of notebook 03 |
| `data/label_queue.json` | Notebook 09's labelling panel |

### The interview angle
> *"How do you evaluate an agent in production, with no ground truth?"*

The reframe: you stop trying to score correctness and start scoring *consistency and shape*.
Faithfulness against the answer's own logged context needs no reference. Trajectory rules are
deterministic and free, so they run on everything. A paired canary turns "is it good?" into
"is it better than what we already shipped?", which is both answerable and the actual
decision. Human labels arrive days later and calibrate the judges — they are not in the
serving path.

The detail that shows operational experience: **name the alert that stays silent.** In the
`retry_loop` drill every quality number is unchanged and customers are being helped correctly —
just three times over, at three times the price. Knowing which of your alerts is blind to which
incident is the difference between an on-call rotation and a dashboard.

In [ ]:
# ============================================================================
# SAVE ARTIFACTS: the online report, and the loop back to notebook 03
# ============================================================================
report = {
    "window": "simulated-day-1",
    "n_conversations": len(TRAFFIC),
    "traffic_mix": {k: round(v / len(TRAFFIC), 4) for k, v in by_intent.items()},
    "online_metrics": {
        "resolution_rate": round(
            statistics.mean(1.0 if e["resolved"] else 0.0 for e in TRAFFIC), 4),
        "citation_rate": round(
            statistics.mean(1.0 if e["cited"] else 0.0 for e in TRAFFIC), 4),
        "cost_per_conversation": round(cost_per_conversation(TRAFFIC), 6),
        "note": ("No reference answers were used. These are not comparable to "
                 "the offline numbers in baseline_report.json."),
    },
    "sampling": {
        "deterministic_coverage": 1.0,
        "judge_sample_rate": JUDGE_SAMPLE_RATE,
        "judged_conversations": len(sampled),
        "estimated_daily_judge_cost_usd": round(
            len(sampled) * JUDGE_COST_PER_CALL, 4),
    },
    "alerts": [{"name": a["name"], "value": a["value"], "limit": a["limit"],
                "fired": a["fired"]} for a in healthy_alerts],
    "alert_coverage": {label: sorted(n for n, f in fired.items() if f)
                       for label, fired in rows},
    "drift": {
        "golden_set_lang_mix": {k: round(v, 4) for k, v in golden_share.items()},
        "production_lang_mix": {k: round(v, 4) for k, v in production_share.items()},
        "total_variation_distance": round(total_variation / 2.0, 4),
    },
    "label_queue": {k: len(v) for k, v in SELECTED.items()},
    "offline_reference": {
        "baseline_resolution_rate": BASELINE["scores"]["resolution_rate"],
        "gate_decision": GATE_RESULT["decision"]["allowed"],
        "golden_set_version": GOLDEN["datasheet"]["version"],
    },
}

report_path = os.path.join(DATA_DIR, "online_report.json")
with open(report_path, "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2)

print("💾 Wrote", report_path)
print("🚨 alerts firing on a healthy day: %d (should be 0)"
      % sum(1 for a in healthy_alerts if a["fired"]))
print("🔁 queued for labelling: %d traces"
      % sum(len(v) for v in SELECTED.values()))
print()
print("✅ Notebook 12 complete — and the course with it.")
print()
print("   The loop closes here: those queued traces go to notebook 09 for human")
print("   labels, become golden cases in notebook 03, and make notebook 11's")
print("   gate harder than it was this morning. That is the only part of an")
print("   evaluation harness that compounds.")